# Object Recognition with Open Images v7 and Webcam

In this notebook we prepare a dataset from **Open Images v7** using **FiftyOne**, crop the annotated objects with **OpenCV**, then adapt a **ResNet50** model with transfer learning and fine-tuning in **TensorFlow/Keras** to recognize everyday objects on a live webcam feed.

**Steps:**
1. Import libraries
2. Choose target classes
3. Download the train and validation subsets
4. Crop objects and check class balance *(next)*
5. Transfer learning with a frozen ResNet50 *(next)*
6. Fine-tuning the top layers *(next)*
7. Evaluate and run the live webcam demo *(next)*

In [1]:
import fiftyone as fo
import fiftyone.zoo as foz

In [6]:
print(fo.list_datasets())

['train_dataset']


## 1. Target classes

We pick **96 everyday classes** that are likely to appear in front of a webcam:

- **Electronics and office:** phone, laptop, keyboard, mouse, monitor, calculator, stapler, ...
- **Kitchen and tableware:** mug, plate, bowl, spoon, fork, knife, teapot, kettle, ...
- **Food and drinks:** fruits, vegetables, fast food, bakery items, sweets, beverages
- **Personal items and home decor:** watch, sunglasses, hat, handbag, lamp, vase, pillow, ...
- **People and animals:** person, human face, human hand, cat, dog

Class names must match the Open Images names exactly (capitalized as in the official list). The next cell prints the number of classes as a sanity check.

In [4]:
labels = ["Mobile phone", "Laptop", "Camera", "Computer keyboard", "Computer mouse", "Headphones",
          "Mug", "Bottle", "Coffee cup", "Book", "Scissors", "Glasses", "Pen", "Clock", "Backpack",
          "Person", "Chair", "Television", "Remote control", "Microphone", "Ice cream", 
          "Grape", "Strawberry", "Pear", "Peach", "Pineapple", "Watermelon", "Mango",
          "Banana", "Apple", "Orange", "Lemon", "Tomato", "Carrot", "Cucumber", "Potato",
          "Pizza", "Hamburger", "Hot dog", "French fries", "Sandwich", "Fast food", "Popcorn", 
          "Bread", "Bagel", "Pretzel", "Muffin", "Croissant", "Cake", "Cookie", "Candy", 
          "Milk", "Juice", "Tea", "Beer", "Wine", "Cocktail","Tablet computer", "Computer monitor", 
          "Calculator", "Ruler", "Stapler", "Eraser", "Pencil case", "Envelope", "Whiteboard",
          "Plate", "Bowl", "Spoon", "Fork", "Knife", "Teapot", "Kettle", "Wine glass", "Tin can",
          "Watch", "Sunglasses", "Hat", "Handbag", "Umbrella", "Toothbrush", "Towel",
          "Lamp", "Picture frame", "Houseplant", "Vase", "Pillow", "Candle", "Mirror",
          "Teddy bear", "Cat", "Dog", "Dumbbell", "Human face", "Human hand", "Salad",]
print(len(labels))

96


## 2. Download the dataset

We download the **train** split of Open Images v7 with bounding-box labels (`detections`), **one class at a time**, so every class gets a similar number of images instead of letting frequent classes (Person, Clothing, ...) dominate.

- `classes=[label]`: only images containing the current class (one class per iteration)
- `max_samples=1000`: images per class (about 96,000 images in total for 96 classes)
- `shuffle=True`, `seed=i`: random but reproducible selection
- `dataset_name`: a temporary dataset per class, merged into `train_dataset` and then deleted

> The first run downloads large metadata files (about 4.8 GB), so it can take several minutes. Later runs use the local cache in `~/fiftyone/open-images-v7`. If the download is interrupted, run the cell again: already downloaded images are not downloaded twice.

In [ ]:
if fo.dataset_exists("train_dataset"):
    fo.delete_dataset("train_dataset")
train_ds = fo.Dataset("train_dataset", persistent=True)

for i, label in enumerate(labels):
    ds = foz.load_zoo_dataset(
        "open-images-v7",
        split="train",
        classes=[label],
        label_types=["detections"],
        max_samples=1000,
        shuffle=True,
        seed=i,
        only_matching=True,
        dataset_name=f"tmp_{i}",
    )
    train_ds.add_samples(ds)
    ds.delete()

 100% |██████|    4.8Gb/4.8Gb [13.7m elapsed, 0s remaining, 5.8Mb/s]      


## 3. Download the validation split

We download the **validation** split of Open Images v7 the same way as the train split: one class at a time, same target classes, same label type (`detections`), so the model can be evaluated on images it has never seen during training.

- `split="validation"`: the validation part of Open Images v7
- `classes=[label]`: only images containing the current class (one class per iteration)
- `max_samples=200`: images per class (about 19,200 images in total, about 20% of the training size)
- `shuffle=True`, `seed=i`: random but reproducible selection
- `dataset_name`: a temporary dataset per class, merged into `validation_dataset` and then deleted

> The train and validation splits are separate in Open Images, so there is no overlap between them. The validation metadata is much smaller than the train metadata, so this step is faster than the previous download. Rare classes may have fewer than 200 images.

In [ ]:
if fo.dataset_exists("validation_dataset"):
    fo.delete_dataset("validation_dataset")
train_ds = fo.Dataset("validation_dataset", persistent=True)

for i, label in enumerate(labels):
    ds = foz.load_zoo_dataset(
        "open-images-v7",
        split="validation",
        classes=[label],
        label_types=["detections"],
        max_samples=200,
        shuffle=True,
        seed=i,
        only_matching=True,
        dataset_name=f"tmp_{i}",
    )
    train_ds.add_samples(ds)
    ds.delete()